In [ ]:
import os
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import PySAM
import PySAM.ResourceTools as tools
import PySAM.Pvwattsv8 as pvwatts
from PySAM.UtilityRateTools import URDBv8_to_ElectricityRates
import PySAM.Utilityrate5 as utility

/Users/rciez/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
OPENEI_API_KEY = "YOUR_OPENEI_KEY"     # free: https://openei.org/services/api/signup/
# you should be able to use the same api key as the NSRDB
URDB_URL = "https://api.openei.org/utility_rates"

# --- 1. List current, approved SCE industrial tariffs ---
params = dict(
    version=8, format="json", api_key=OPENEI_API_KEY,
    ratesforutility="Pacific Gas & Electric Co",   # utility name as it appears in URDB
    sector="Industrial",
    approved="true",
    detail="minimal",
    orderby="startdate", direction="desc",
    limit=500,
)
r = requests.get(URDB_URL, params=params, timeout=60)
r.raise_for_status()
items = r.json()["items"]

tariffs = pd.DataFrame(items)
#if "enddate" in tariffs.columns:
#    tariffs = tariffs[tariffs["enddate"].isna()]                 # keep tariffs still in effect
tariffs["startdate"] = pd.to_datetime(tariffs["startdate"], unit="s")
pd.set_option("display.max_colwidth", 90)
print(f"{len(tariffs)} current PG&E industrial tariffs")
tariffs[["label", "name", "startdate"]].sort_values("name")

4 current PG&E industrial tariffs


,label,name,startdate
3,539f6ef0ec4f024411ecc871,E-20 (Primary),2014-01-01 08:00:00
0,574dbd695457a30e7e5e629f,E-20 Maximum demand of (1000 KW or more) (Primary),2016-03-24 07:00:00
1,555385a35457a3012d8b456a,E-20 Maximum demand of (1000 KW or more) (Primary),2015-03-01 08:00:00
2,5400ac345257a3e73199cb96,E-20 Maximum demand of (1000 KW or more) (Primary),2014-05-01 07:00:00


In [4]:
# --- 2. Pick a tariff and download its full rate structure ---
# Copy a label from the table above. Large SCE industrial customers (>500 kW) are on TOU-8;
# pick the version that matches the service voltage (e.g., "below 2 kV" for a typical factory).
LABEL = "PASTE_LABEL_HERE"
LABEL = "574dbd695457a30e7e5e629f"

r = requests.get(URDB_URL, params=dict(version=8, format="json", api_key=OPENEI_API_KEY,
                                       getpage=LABEL, detail="full"), timeout=60)
r.raise_for_status()
tariff = r.json()["items"][0]
print(tariff["name"])
print("Effective:", pd.to_datetime(tariff["startdate"], unit="s").date())
print("Source:   ", tariff.get("source", "n/a"))

# --- 3. Convert to PySAM inputs and rerun the bill calculation ---
rates = URDBv8_to_ElectricityRates(tariff)

E-20 Maximum demand of (1000 KW or more) (Primary)
Effective: 2016-03-24
Source:    http://www.pge.com/tariffs/tm2/pdf/ELEC_SCHEDS_E-20.pdf


In [ ]:
ur_real = utility.default("PVWattsCommercial")
ur_real.SystemOutput.gen = pv_ind.Outputs.gen          # 1 MW PV from the previous cell
ur_real.SystemOutput.degradation = [0]
ur_real.Lifetime.analysis_period = 1
ur_real.Lifetime.inflation_rate = 0
ur_real.Load.load = list(load_kw)                      # factory load from the previous cell
ur_real.ElectricityRates.assign(rates)
# URDB often leaves out net-metering rules, and the converter then assumes net metering.
# Uncomment to model net billing (exports at the sell rate) instead:
# ur_real.ElectricityRates.ur_metering_option = 2
ur_real.execute()
o2 = ur_real.Outputs

summary_real = pd.DataFrame({
    name: {"Without PV": sum(getattr(o2, wo)), "With PV": sum(getattr(o2, w))}
    for name, (wo, w) in parts.items()
}).T
summary_real.loc["Total"] = summary_real.sum()
summary_real["Savings"] = summary_real["Without PV"] - summary_real["With PV"]
print(f"Average cost without PV: ${summary_real.loc['Total', 'Without PV'] / load_kw.sum():.3f}/kWh")
display(summary_real.round(0).astype(int).style.format("${:,.0f}"))